# Установка библиотек

In [6]:
!pip install ultralytics albumentations

  Using cached albumentations-2.0.8-py3-none-any.whl.metadata (43 kB)
  Using cached albucore-0.0.24-py3-none-any.whl.metadata (5.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.7 MB/s  0:00:00
Using cached albumentations-2.0.8-py3-none-any.whl (369 kB)
Using cached albucore-0.0.24-py3-none-any.whl (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.8/865.8 kB 11.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 11.2 MB/s  0:00:04 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 583.6/583.6 kB 11.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 11.6 MB/s  0:00:001.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [ultralytics]m 10/11 [ultralytics]ns]tform]-32]


# Импорты

In [1]:
import os
import json
import time
from datetime import datetime, timedelta
from pathlib import Path

import yaml
import cv2
import numpy as np
from ultralytics import YOLO
import albumentations as A
import torch
from PIL import Image, ImageDraw, ImageFont

/home/himoza/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Конфигурация

In [2]:
RESULTS_DIR = "./train_result"
os.makedirs(RESULTS_DIR, exist_ok=True)

DATA_YAML = "./block-detection-1/data.yaml"

DEVICE = 0 if torch.cuda.is_available() else 'cpu'

EPOCHS = 200
IMGSZ = 640
PATIENCE = 15          # Early stopping: остановка через 10 эпох без улучшения
BATCH_SIZE = 32

## Функция обучения

In [3]:
def train_yolo(
    model_name: str,
    project_name: str,
    results_dir: str,
    data_yaml: str,
    epochs: str,
    batch_size: int,
    imgsz: int,
    patience: int,
    custom_augmentations: list = None
  ):
    """
      Обучение YOLO модели без аугментации с отслеживанием времени

      Args:
          model_name: путь к модели или имя ('yolov8n.pt', etc.)
          project_name: имя проекта
          results_dir: директория для сохранения результатов
          data_yaml: путь к data.yaml
          epochs: количество эпох
          batch_size: размер батча
          imgsz: размер изображения
          patience: early stopping patience
          custom_augmentations: Список аугментаций

      Returns:
          results: результаты обучения
          training_time: время обучения в секундах
    """
    # Загрузка модели
    model = YOLO(model_name,)
    full_project_path = os.path.abspath(os.path.join(results_dir, project_name))

    # Время начала обучения
    start_time = time.time()
    start_datetime = datetime.now()

    print("\n" + "="*60)
    print(f"НАЧАЛО ОБУЧЕНИЯ: {project_name}")
    print("="*60)
    print(f"Время начала: {start_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Путь сохранения: {full_project_path}")
    print(f"Эпохи: {epochs} | Patience: {patience}")
    print("="*60 + "\n")

    # Обучение
    results = model.train(
        data=data_yaml,
        epochs=epochs,
        batch=batch_size,
        imgsz=imgsz,
        device=DEVICE,
        project=full_project_path,   # абсолютный путь
        name="train",
        exist_ok=True,                  # Перезаписывать, если существует
        patience=patience,              # Встроенный patience в ultralytics
        save=True,                      # Сохранять веса
        save_period=10,                 # Сохранять каждые 10 эпох
        val=True,                       # Валидация после каждой эпохи
        plots=True,                     # Сохранять графики
        verbose=True,
        augmentations=custom_augmentations
    )

    # Время окончания обучения
    end_time = time.time()
    end_datetime = datetime.now()
    training_time_seconds = end_time - start_time

    # Форматирование времени
    training_time_formatted = str(timedelta(seconds=int(training_time_seconds)))
    hours = int(training_time_seconds // 3600)
    minutes = int((training_time_seconds % 3600) // 60)
    seconds = int(training_time_seconds % 60)

    # Сохранение конфигурации обучения
    config_info = {
        "project": project_name,
        "data_yaml": data_yaml,
        "epochs": epochs,
        "patience": patience,
        "batch_size": batch_size,
        "imgsz": imgsz,
        "device": str(DEVICE),
        "training_time": {
            "seconds": training_time_seconds,
            "formatted": training_time_formatted,
            "hours": hours,
            "minutes": minutes,
            "seconds": seconds
        },
        "start_time": start_datetime.isoformat(),
        "end_time": end_datetime.isoformat(),
        "total_epochs_completed": getattr(results, 'epoch', epochs) if hasattr(results, 'epoch') else epochs
    }

    # Вывод информации о времени обучения
    print("\n" + "="*60)
    print("ОБУЧЕНИЕ ЗАВЕРШЕНО")
    print("="*60)
    print(f"Время начала:   {start_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Время окончания: {end_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Общее время:     {training_time_formatted} ({hours}ч {minutes}м {seconds}с)")
    print(f"Результаты:      {full_project_path}")
    print("="*60 + "\n")
    return results

## Функция валидации

In [4]:
def validate_best_model(
    project_name: str,
    results_dir: str,
    data_yaml: str,
    epochs: str,
    batch_size: int,
    imgsz: int
  ):
    """Валидация лучшей модели и сохранение метрик"""
    full_project_path = os.path.abspath(os.path.join(results_dir, project_name))
    best_model_path = os.path.join(full_project_path, 'train', 'weights', 'best.pt')

    if not os.path.exists(best_model_path):
        print("Best model not found!")
        return None

    model = YOLO(best_model_path)

    # Валидация с подробными метриками
    metrics = model.val(
        data=data_yaml,
        batch=batch_size,
        imgsz=imgsz,
        device=DEVICE,
        plots=True,
        save_json=True,
        save_hybrid=True,
        project=full_project_path,
        name="val", 
    )

    # Сохранение метрик в текстовый файл
    metrics_path = os.path.join(full_project_path, "validation_metrics.txt")
    with open(metrics_path, 'w') as f:
        f.write("="*50 + "\n")
        f.write("РЕЗУЛЬТАТЫ ВАЛИДАЦИИ МОДЕЛИ\n")
        f.write("="*50 + "\n\n")

        # Основные метрики
        f.write("ОСНОВНЫЕ МЕТРИКИ:\n")
        f.write(f"  mAP50-95: {metrics.box.map:.4f}\n")
        f.write(f"  mAP50:     {metrics.box.map50:.4f}\n")
        f.write(f"  mAP75:     {metrics.box.map75:.4f}\n\n")

        # Дополнительные метрики (если доступны)
        if hasattr(metrics.box, 'mp'):
            f.write("ДОПОЛНИТЕЛЬНЫЕ МЕТРИКИ:\n")
            f.write(f"  Precision: {metrics.box.mp:.4f}\n")
            f.write(f"  Recall:    {metrics.box.mr:.4f}\n\n")

        # Метрики по классам
        f.write("МЕТРИКИ ПО КЛАССАМ:\n")
        if hasattr(metrics.box, 'ap') and hasattr(metrics.box, 'ap_class_index'):
            for cls_id, ap_value in zip(metrics.box.ap_class_index, metrics.box.ap):
                f.write(f"  Class {cls_id}: mAP = {ap_value:.4f}\n")

        # Сводка
        f.write("\n" + "="*50 + "\n")
        f.write(f"Всего классов: {len(metrics.box.ap_class_index)}\n")
        f.write(f"Лучший класс: Class {metrics.box.ap_class_index[metrics.box.ap.argmax()]} "
                f"(mAP={metrics.box.ap.max():.4f})\n")
        f.write(f"Худший класс: Class {metrics.box.ap_class_index[metrics.box.ap.argmin()]} "
                f"(mAP={metrics.box.ap.min():.4f})\n")

    print(f"Validation metrics saved to: {metrics_path}")
    print(f"\nИтоговые метрики:")
    print(f"   mAP50-95: {metrics.box.map:.4f}")
    print(f"   mAP50: {metrics.box.map50:.4f}")
    print(f"   mAP75: {metrics.box.map75:.4f}")

    return metrics


## Функция сопоставления предсказания YOLO с ground truth разметкой

In [5]:
def visualize_predictions_vs_gt(
    project_name,
    results_dir,
    data_yaml_path,
    num_samples=20,
    conf_threshold=0.25,
    iou_threshold=0.45,
    img_size=640,
    seed=42
):
    """
    Создает изображения сравнения ground truth и предсказаний YOLO

    Args:
        project_name: имя проекта
        results_dir: путь к результату
        data_yaml_path: путь к data.yaml с информацией о датасете
        num_samples: количество изображений для визуализации
        conf_threshold: порог уверенности для предсказаний
        iou_threshold: порог IoU для NMS
        img_size: размер изображения для модели
        seed: random seed для воспроизводимости
    """
    full_project_path = os.path.join(results_dir, project_name)
    best_model_path = os.path.join(full_project_path, 'train', 'weights', 'best.pt')
    output_dir = os.path.join(full_project_path, 'val', 'res')
    # Создаем выходную директорию
    os.makedirs(output_dir, exist_ok=True)

    # Загружаем модель
    model = YOLO(best_model_path)

    # Загружаем конфигурацию датасета
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)

    class_names = data_config['names']
    num_classes = len(class_names)

    # Генерируем цвета для классов (фиксированные для воспроизводимости)
    np.random.seed(42)
    colors = np.random.randint(0, 255, size=(num_classes, 3), dtype=np.uint8)

    # Загружаем список изображений из валидационного сета
    val_images_dir = data_config.get('val', data_config.get('valid'))[3:]

    if val_images_dir is None:
        raise ValueError("Не найден путь к валидационным изображениям в data.yaml")

    # Получаем список изображений
    if os.path.isabs(val_images_dir):
        val_path = Path(val_images_dir)
    else:
        val_path = Path(data_yaml_path).parent / val_images_dir

    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
        image_files.extend(list(val_path.glob(ext)))

    print(val_path)
    if len(image_files) == 0:
        raise ValueError(f"Не найдены изображения в {val_path}")

    # Случайно выбираем num_samples изображений
    np.random.seed(seed)
    selected_images = np.random.choice(image_files,
                                       size=min(num_samples, len(image_files)),
                                       replace=False)

    # Функция для отрисовки боксов (GT или предсказания)
    def draw_boxes(img, boxes, labels=None, confidences=None, is_pred=False):
        """
        Рисует bounding boxes на изображении
        """
        img_copy = img.copy()
        h, w = img_copy.shape[:2]

        for i, box in enumerate(boxes):
            # Конвертируем нормализованные координаты в пиксельные
            if max(box) <= 1.0:  # если координаты нормализованы
                x1, y1, x2, y2 = box
                x1, y1 = int(x1 * w), int(y1 * h)
                x2, y2 = int(x2 * w), int(y2 * h)
            else:
                x1, y1, x2, y2 = map(int, box)

            # Определяем цвет и параметры
            if is_pred:
                # Для предсказаний используем цвет класса и пунктир
                if labels is not None and i < len(labels):
                    class_id = int(labels[i])
                    color = tuple(map(int, colors[class_id % len(colors)]))
                else:
                    color = (0, 255, 0)  # зеленый по умолчанию

                thickness = 2
                line_type = cv2.LINE_AA

                # Рисуем прямоугольник
                cv2.rectangle(img_copy, (x1, y1), (x2, y2), color, thickness, line_type)

                # Добавляем подпись с классом и уверенностью
                if labels is not None and i < len(labels):
                    class_id = int(labels[i])
                    class_name = class_names[class_id] if class_id < len(class_names) else f"Class {class_id}"

                    conf_str = ""
                    if confidences is not None and i < len(confidences):
                        conf_str = f" {confidences[i]:.2f}"

                    label_text = f"{class_name}{conf_str}"

                    # Рисуем фон для текста
                    (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                    cv2.rectangle(img_copy, (x1, y1 - text_h - 5), (x1 + text_w, y1), color, -1)
                    cv2.putText(img_copy, label_text, (x1, y1 - 3),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)
            else:
                # Для GT используем синий цвет и сплошную линию
                color = (255, 0, 0)  # синий для GT
                thickness = 2
                line_type = cv2.LINE_AA

                cv2.rectangle(img_copy, (x1, y1), (x2, y2), color, thickness, line_type)

                # Добавляем подпись класса для GT
                if labels is not None and i < len(labels):
                    class_id = int(labels[i])
                    class_name = class_names[class_id] if class_id < len(class_names) else f"Class {class_id}"

                    label_text = class_name
                    (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                    cv2.rectangle(img_copy, (x1, y1 - text_h - 5), (x1 + text_w, y1), color, -1)
                    cv2.putText(img_copy, label_text, (x1, y1 - 3),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)

        return img_copy

    # Обрабатываем каждое изображение
    for idx, img_path in enumerate(selected_images):
        print(f"Обработка {idx+1}/{len(selected_images)}: {img_path.name}")

        # Загружаем изображение
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"Не удалось загрузить изображение: {img_path}")
            continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # === Получаем Ground Truth ===
        # Ищем соответствующий label файл
        label_path = None
        possible_label_dirs = [
            val_path.parent / 'labels' / val_path.name,  # если labels/val
            Path(data_config.get('labels', '')) / val_path.name if data_config.get('labels') else None,
            val_path.parent / 'labels'  # если labels в той же папке
        ]

        for label_dir in possible_label_dirs:
            if label_dir is not None and label_dir.exists():
                potential_label = label_dir / f"{img_path.stem}.txt"
                if potential_label.exists():
                    label_path = potential_label
                    break

        # Если не нашли в стандартных местах, ищем рядом с изображением
        if label_path is None:
            potential_label = img_path.parent / f"{img_path.stem}.txt"
            if potential_label.exists():
                label_path = potential_label

        gt_boxes = []
        gt_labels = []

        if label_path and label_path.exists():
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        class_id = int(parts[0])
                        # YOLO формат: class_id cx cy w h (все нормализовано)
                        cx, cy, bw, bh = map(float, parts[1:5])
                        x1 = cx - bw / 2
                        y1 = cy - bh / 2
                        x2 = cx + bw / 2
                        y2 = cy + bh / 2

                        gt_boxes.append([x1, y1, x2, y2])
                        gt_labels.append(class_id)

        # === Получаем предсказания ===
        results = model.predict(
            str(img_path),
            conf=conf_threshold,
            iou=iou_threshold,
            imgsz=img_size,
            verbose=False
        )

        pred_boxes = []
        pred_labels = []
        pred_confs = []

        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            if len(boxes) > 0:
                # Получаем координаты в формате xyxy
                xyxy_boxes = boxes.xyxy.cpu().numpy()
                class_ids = boxes.cls.cpu().numpy()
                confidences = boxes.conf.cpu().numpy()

                # Нормализуем координаты
                h, w = img.shape[:2]
                normalized_boxes = []
                for box in xyxy_boxes:
                    x1, y1, x2, y2 = box
                    normalized_boxes.append([x1/w, y1/h, x2/w, y2/h])

                pred_boxes = normalized_boxes
                pred_labels = class_ids.tolist()
                pred_confs = confidences.tolist()

        # === Создаем изображения с аннотациями ===
        gt_img = draw_boxes(img_rgb, gt_boxes, gt_labels, is_pred=False)
        pred_img = draw_boxes(img_rgb, pred_boxes, pred_labels, pred_confs, is_pred=True)

        # === Объединяем изображения side-by-side ===
        # Добавляем заголовки
        gt_img_pil = Image.fromarray(gt_img)
        pred_img_pil = Image.fromarray(pred_img)

        # Создаем объединенное изображение
        total_width = gt_img_pil.width + pred_img_pil.width
        max_height = max(gt_img_pil.height, pred_img_pil.height) + 40  # +40 для заголовка

        combined_img = Image.new('RGB', (total_width, max_height), color=(255, 255, 255))

        # Вставляем изображения
        combined_img.paste(gt_img_pil, (0, 40))
        combined_img.paste(pred_img_pil, (gt_img_pil.width, 40))

        # Добавляем текст заголовков
        draw = ImageDraw.Draw(combined_img)

        try:
            # Пытаемся использовать системный шрифт
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 20)
        except:
            # Если шрифт не найден, используем стандартный
            font = ImageFont.load_default()

        # Заголовки
        draw.text((10, 10), "Ground Truth", fill=(255, 0, 0), font=font)
        draw.text((gt_img_pil.width + 10, 10), "Predictions", fill=(0, 255, 0), font=font)

        # Дополнительная информация: название файла и статистика
        stats_text = f"File: {img_path.name} | GT objects: {len(gt_boxes)} | Pred objects: {len(pred_boxes)}"
        draw.text((10, max_height - 25), stats_text, fill=(0, 0, 0), font=ImageFont.load_default())

        # Сохраняем результат
        output_filename = f"comparison_{idx+1:03d}_{img_path.stem}.jpg"
        output_path = os.path.join(output_dir, output_filename)
        combined_img.save(output_path, quality=95)

        print(f"  Сохранено: {output_filename}")
        print(f"  GT: {len(gt_boxes)} объектов, Predictions: {len(pred_boxes)} объектов")
        print()

    print(f"\n✅ Готово! Все изображения сохранены в: {output_dir}")
    print(f"Всего обработано изображений: {len(selected_images)}")

    return output_dir

# Обучение без аугментаций

## YOLOv8n

### Конфигурация

In [13]:
model = 'yolov8n.pt'
project_name = "yolov8n_training"

### Обучение

In [14]:
train_results = train_yolo(
    model,
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    EPOCHS,
    BATCH_SIZE,
    IMGSZ,
    PATIENCE
  )


НАЧАЛО ОБУЧЕНИЯ: yolov8n_training
Время начала: 2026-09-30 20:27:44
Путь сохранения: ./train_result/yolov8n_training
Эпохи: 200 | Patience: 15

Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=None, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./block-detection-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01,

FileNotFoundError: [Errno 2] No such file or directory: './train_result/yolov8n_training/training_config.json'

### Валидация

In [20]:
metrics = validate_best_model(
      project_name,
      RESULTS_DIR,
      DATA_YAML,
      EPOCHS,
      BATCH_SIZE,
      IMGSZ,
    )

WARNING ⚠️ 'save_hybrid' is deprecated and will be removed in the future.


Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2813.4±413.0 MB/s, size: 77.6 KB)
val: Scanning /home/himoza/block_detect/block-detection-1/valid/labels.cache... 123 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 123/123 32.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.1s/it 4.5s1.5s
                   all        123       8279       0.96      0.964      0.988       0.83
Speed: 3.3ms preprocess, 19.6ms inference, 0.0ms loss, 4.6ms postprocess per image
Saving /home/himoza/block_detect/train_result/yolov8n_training/val/predictions.json...

Evaluating faster-coco-eval mAP...
Evaluate annotation type *bbox*
COCOeval_opt.evaluate() finished...
DONE (t=0.10s).
Accumulating evaluation results...
COCOeval_opt.accumulate() finished...

In [21]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    num_samples=30,
    conf_threshold=0.25,
    iou_threshold=0.45,
    img_size=640,
    seed=42
)

block-detection-1/valid/images
Обработка 1/30: 6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  Сохранено: comparison_001_6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  GT: 24 объектов, Predictions: 24 объектов

Обработка 2/30: artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  Сохранено: comparison_002_artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  GT: 36 объектов, Predictions: 36 объектов

Обработка 3/30: wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  Сохранено: comparison_003_wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  GT: 40 объектов, Predictions: 40 объектов

Обработка 4/30: 4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  Сохранено: comparison_004_4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  GT: 104 объектов, Predictions: 104 объектов

Обработка 5/30: _barhat-2-500x500-500x500_jpg.rf.

'./train_result/yolov8n_training/val/res'

## YOLOv11n

### Конфигурация

In [6]:
model = 'yolo11n.pt'
project_name = "yolov11n_training"

### Обучение

In [ ]:
train_results = train_yolo(
    model,
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    EPOCHS,
    BATCH_SIZE,
    IMGSZ,
    PATIENCE
  )


НАЧАЛО ОБУЧЕНИЯ: yolov11n_training
Время начала: 2026-09-30 20:55:37
Путь сохранения: /home/himoza/block_detect/train_result/yolov11n_training
Эпохи: 200 | Patience: 15

Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=None, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./block-detection-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width

### Валидация

In [7]:
metrics = validate_best_model(
      project_name,
      RESULTS_DIR,
      DATA_YAML,
      EPOCHS,
      BATCH_SIZE,
      IMGSZ
    )

WARNING ⚠️ 'save_hybrid' is deprecated and will be removed in the future.
Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 181.9±42.0 MB/s, size: 66.7 KB)
val: Scanning /home/himoza/block_detect/block-detection-1/valid/labels.cache... 123 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 123/123 32.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.2s/it 4.9s1.6s
                   all        123       8279      0.962      0.969       0.99      0.841
Speed: 3.5ms preprocess, 21.0ms inference, 0.0ms loss, 4.5ms postprocess per image
Saving /home/himoza/block_detect/train_result/yolov11n_training/val/predictions.json...

Evaluating faster-coco-eval mAP...
Evaluate annotation type *bbox*
COCOeval_opt.evaluate() finished...
DONE (t=0.11

In [8]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    num_samples=30,
    conf_threshold=0.25,
    iou_threshold=0.45,
    img_size=640,
    seed=42
)

block-detection-1/valid/images
Обработка 1/30: 6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  Сохранено: comparison_001_6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  GT: 24 объектов, Predictions: 24 объектов

Обработка 2/30: artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  Сохранено: comparison_002_artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  GT: 36 объектов, Predictions: 36 объектов

Обработка 3/30: wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  Сохранено: comparison_003_wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  GT: 40 объектов, Predictions: 40 объектов

Обработка 4/30: 4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  Сохранено: comparison_004_4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  GT: 104 объектов, Predictions: 104 объектов

Обработка 5/30: _barhat-2-500x500-500x500_jpg.rf.

'./train_result/yolov11n_training/val/res'

## YOLOv8s

### Конфигурация

In [9]:
model = 'yolov8s.pt'
project_name = "yolov8s_training"

### Обучение

In [10]:
train_results = train_yolo(
    model,
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    EPOCHS,
    BATCH_SIZE,
    IMGSZ,
    PATIENCE
  )


НАЧАЛО ОБУЧЕНИЯ: yolov8s_training
Время начала: 2026-09-30 21:23:34
Путь сохранения: /home/himoza/block_detect/train_result/yolov8s_training
Эпохи: 200 | Patience: 15

Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=None, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./block-detection-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=N

### Валидация

In [11]:
metrics = validate_best_model(
      project_name,
      RESULTS_DIR,
      DATA_YAML,
      EPOCHS,
      BATCH_SIZE,
      IMGSZ,
    )

WARNING ⚠️ 'save_hybrid' is deprecated and will be removed in the future.
Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)


Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 204.2±89.0 MB/s, size: 58.6 KB)
val: Scanning /home/himoza/block_detect/block-detection-1/valid/labels.cache... 123 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 123/123 51.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.3s/it 5.1s1.7ss
                   all        123       8279      0.961      0.971      0.989      0.832
Speed: 3.6ms preprocess, 17.5ms inference, 0.0ms loss, 9.7ms postprocess per image
Saving /home/himoza/block_detect/train_result/yolov8s_training/val/predictions.json...

Evaluating faster-coco-eval mAP...
Evaluate annotation type *bbox*
COCOeval_opt.evaluate() finished...
DONE (t=0.13s).
Accumulating evaluation results...
COCOeval_opt.accumulate() finished...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.710

## YOLOv11s

### Конфигурация

In [12]:
model = 'yolo11s.pt'
project_name = "yolov11s_training"

### Обучение

In [13]:
train_results = train_yolo(
    model,
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    EPOCHS,
    BATCH_SIZE,
    IMGSZ,
    PATIENCE
  )


НАЧАЛО ОБУЧЕНИЯ: yolov11s_training
Время начала: 2026-09-30 21:41:19
Путь сохранения: /home/himoza/block_detect/train_result/yolov11s_training
Эпохи: 200 | Patience: 15

Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=None, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./block-detection-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width

### Валидация

In [14]:
metrics = validate_best_model(
      project_name,
      RESULTS_DIR,
      DATA_YAML,
      EPOCHS,
      BATCH_SIZE,
      IMGSZ
    )

WARNING ⚠️ 'save_hybrid' is deprecated and will be removed in the future.


Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2180.9±814.3 MB/s, size: 58.6 KB)
val: Scanning /home/himoza/block_detect/block-detection-1/valid/labels.cache... 123 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 123/123 57.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.2s/it 4.9s1.6ss
                   all        123       8279      0.962      0.971      0.989      0.839
Speed: 3.6ms preprocess, 21.7ms inference, 0.0ms loss, 4.1ms postprocess per image
Saving /home/himoza/block_detect/train_result/yolov11s_training/val/predictions.json...

Evaluating faster-coco-eval mAP...
Evaluate annotation type *bbox*
COCOeval_opt.evaluate() finished...
DONE (t=0.12s).
Accumulating evaluation results...
COCOeval_opt.accumulate() finis

In [15]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    num_samples=30,
    conf_threshold=0.25,
    iou_threshold=0.45,
    img_size=640,
    seed=42
)

block-detection-1/valid/images
Обработка 1/30: 6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg


  Сохранено: comparison_001_6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  GT: 24 объектов, Predictions: 24 объектов

Обработка 2/30: artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  Сохранено: comparison_002_artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  GT: 36 объектов, Predictions: 36 объектов

Обработка 3/30: wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  Сохранено: comparison_003_wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  GT: 40 объектов, Predictions: 40 объектов

Обработка 4/30: 4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  Сохранено: comparison_004_4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  GT: 104 объектов, Predictions: 104 объектов

Обработка 5/30: _barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a75454ad8649138.jpg
  Сохранено: comparison_005__barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a75454ad8

'./train_result/yolov11s_training/val/res'

# Обучение с аугментаций

## Аугментации

In [6]:
custom_augmentations = A.Compose([

    # ГЕОМЕТРИЯ (минимальная)
    A.Affine(
        translate_percent=(-0.03, 0.03),
        scale=(0.9, 1.1),
        p=0.5,
    ),

    A.HorizontalFlip(p=0.5),

    # ОСВЕЩЕНИЕ
    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.6,
    ),

    A.CLAHE(
        clip_limit=2.0,
        tile_grid_size=(8, 8),
        p=0.3,
    ),

    # ЦВЕТ
    A.HueSaturationValue(
        hue_shift_limit=5,
        sat_shift_limit=15,
        val_shift_limit=10,
        p=0.3,
    ),

    # ШУМ
    A.GaussNoise(
        std_range=(0.02, 0.08),
        p=0.3,
    ),

    # РАЗМЫТИЕ
    A.GaussianBlur(
        blur_limit=(3, 3),
        p=0.1,
    ),

    # ЧАСТИЧНЫЕ ПЕРЕКРЫТИЯ
    A.CoarseDropout(
        num_holes_range=(1, 2),
        hole_height_range=(8, 16),
        hole_width_range=(8, 16),
        fill=114,
        p=0.2,
    ),

],
bbox_params=A.BboxParams(
    format='yolo',
    label_fields=['class_labels'],
))




## YOLOv26n

### Конфигурация

In [7]:
model = 'yolo26n.pt'
project_name = "aug_yolo26n_training"
BATCH_SIZE = 25

### Обучение

In [8]:
train_results = train_yolo(
    model,
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    EPOCHS,
    BATCH_SIZE,
    IMGSZ,
    PATIENCE,
    custom_augmentations
  )


НАЧАЛО ОБУЧЕНИЯ: aug_yolo26n_training
Время начала: 2026-09-30 22:20:00
Путь сохранения: /home/himoza/block_detect/train_result/aug_yolo26n_training
Эпохи: 200 | Patience: 15

New https://pypi.org/project/ultralytics/8.4.169 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[{'__version__': '2.0.8', 'transform': {'__class_fullname__': 'Affine', 'p': 0.5, 'balanced_scale': False, 'border_mode': 0, 'fill': 0.0, 'fill_mask': 0.0, 'fit_output': False, 'interpolation': 1, 'keep_ratio': False, 'mask_interpolation': 0, 'rotate': (0.0, 0.0), 'rotate_method': 'largest_box', 'scale': {'x': (0.9, 1.1), 'y': (0.9, 1.1)}, 'shear': {'x': (0.0, 0.0), 'y': (0.0, 0.0)}, 'translate_percent': {'x': (-0.03, 0.03), 'y': (-0.03, 0.03)}, 'translate_px': None}}, {'__version__': '2.0.8', 'transform': {'__class_fullname__': 'Ho

/home/himoza/miniconda3/lib/python3.13/site-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas



      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      1/200      4.79G      2.007      3.324    0.01169       2327        640: 3% ──────────── 1/28 1.8s/it 8.3s<48.0sWARNING ⚠️ CUDA out of memory in TaskAlignedAssigner with batch_size=25 and max_num_obj=664; retrying assignment one image at a time on GPU. Model forward batch size is unchanged.
WARNING ⚠️ CUDA out of memory in TaskAlignedAssigner with batch_size=25 and max_num_obj=664; retrying assignment one image at a time on GPU. Model forward batch size is unchanged.
      1/200       4.8G      1.783      3.268   0.009072       2141        640: 42% ━━━━━─────── 12/28 2.7it/s 13.0s<6.0s

/home/himoza/miniconda3/lib/python3.13/site-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas


      1/200      4.51G      1.458          3   0.007122       1264        640: 75% ━━━━━━━━━─── 21/28 3.1it/s 16.1s<2.3s

...

      4/200      4.61G      1.015     0.9621   0.004239       2367        640: 100% ━━━━━━━━━━━━ 28/28 3.0it/s 9.2s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.1it/s 1.4s1.1s
                   all        123       8279      0.927      0.955       0.96      0.716

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      5/200      4.62G      1.037      0.891   0.004485       2108        640: 28% ━━━───────── 8/28 3.1it/s 2.9s<6.5ss

/home/himoza/miniconda3/lib/python3.13/site-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas


      5/200      4.53G      1.019     0.8562   0.004428       1975        640: 78% ━━━━━━━━━─── 22/28 3.3it/s 7.8s<1.8s

/home/himoza/miniconda3/lib/python3.13/site-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas


      5/200      4.43G      1.012     0.8489   0.004452       2183        640: 100% ━━━━━━━━━━━━ 28/28 3.0it/s 9.2s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.1it/s 1.4s1.1s
                   all        123       8279      0.926      0.951      0.966       0.74

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      6/200       4.4G      0.994     0.7816   0.004214       2148        640: 100% ━━━━━━━━━━━━ 28/28 2.8it/s 10.1s0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.2it/s 1.4s1.1s
                   all        123       8279       0.94       0.95      0.974      0.749

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      7/200      4.26G     0.9396     0.7243   0.004089       1266        640: 100% ━━━━━━━━━━━━ 28/28 2.9it/s 9.6s0.3s
                 Class     Images  Instances 

### Валидация

In [9]:
metrics = validate_best_model(
      project_name,
      RESULTS_DIR,
      DATA_YAML,
      EPOCHS,
      BATCH_SIZE,
      IMGSZ,
    )

WARNING ⚠️ 'save_hybrid' is deprecated and will be removed in the future.
Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)


YOLO26n summary (fused): 120 layers, 2,375,031 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2710.8±892.8 MB/s, size: 84.0 KB)
val: Scanning /home/himoza/block_detect/block-detection-1/valid/labels.cache... 123 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 123/123 73.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.0it/s 5.0s1.5ss
                   all        123       8279      0.969      0.966       0.99      0.827
Speed: 3.6ms preprocess, 22.7ms inference, 0.0ms loss, 4.5ms postprocess per image
Saving /home/himoza/block_detect/train_result/aug_yolo26n_training/val/predictions.json...

Evaluating faster-coco-eval mAP...
Evaluate annotation type *bbox*
COCOeval_opt.evaluate() finished...
DONE (t=0.12s).
Accumulating evaluation results...
COCOeval_opt.accumulate() finished...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] 

In [10]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    num_samples=30,
    conf_threshold=0.25,
    iou_threshold=0.45,
    img_size=640,
    seed=42
)

block-detection-1/valid/images
Обработка 1/30: 6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  Сохранено: comparison_001_6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  GT: 24 объектов, Predictions: 24 объектов

Обработка 2/30: artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  Сохранено: comparison_002_artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  GT: 36 объектов, Predictions: 36 объектов

Обработка 3/30: wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  Сохранено: comparison_003_wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  GT: 40 объектов, Predictions: 40 объектов

Обработка 4/30: 4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  Сохранено: comparison_004_4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  GT: 104 объектов, Predictions: 104 объектов

Обработка 5/30: _barhat-2-500x500-500x500_jpg.rf.

'./train_result/aug_yolo26n_training/val/res'

## YOLOv26s

### Конфигурация

In [11]:
model = 'yolo26s.pt'
project_name = "aug_yolo26s_training"

### Обучение

In [12]:
train_results = train_yolo(
    model,
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    EPOCHS,
    BATCH_SIZE,
    IMGSZ,
    PATIENCE,
    custom_augmentations
  )


НАЧАЛО ОБУЧЕНИЯ: aug_yolo26s_training
Время начала: 2026-09-30 22:44:29
Путь сохранения: /home/himoza/block_detect/train_result/aug_yolo26s_training
Эпохи: 200 | Patience: 15

New https://pypi.org/project/ultralytics/8.4.170 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[{'__version__': '2.0.8', 'transform': {'__class_fullname__': 'Affine', 'p': 0.5, 'balanced_scale': False, 'border_mode': 0, 'fill': 0.0, 'fill_mask': 0.0, 'fit_output': False, 'interpolation': 1, 'keep_ratio': False, 'mask_interpolation': 0, 'rotate': (0.0, 0.0), 'rotate_method': 'largest_box', 'scale': {'x': (0.9, 1.1), 'y': (0.9, 1.1)}, 'shear': {'x': (0.0, 0.0), 'y': (0.0, 0.0)}, 'translate_percent': {'x': (-0.03, 0.03), 'y': (-0.03, 0.03)}, 'translate_px': None}}, {'__version__': '2.0.8', 'transform': {'__class_fullname__': 'Ho

/home/himoza/miniconda3/lib/python3.13/site-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas


WARNING ⚠️ CUDA out of memory with batch=25. Reducing to batch=12 and retrying (1/3).
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1802.8±762.2 MB/s, size: 66.2 KB)
train: Scanning /home/himoza/block_detect/block-detection-1/train/labels.cache... 698 images, 0 backgrounds, 2 corrupt: 100% ━━━━━━━━━━━━ 698/698 195.2Mit/s 0.0s
train: /home/himoza/block_detect/block-detection-1/train/images/images124_jpg.rf.17c9b16c193a70c0399325fd995eda13.jpg: ignoring corrupt image/label: labels mix segment and detection rows
train: /home/himoza/block_detect/block-detection-1/train/images/images163_jpg.rf.1933bcd0ff9749f4c5f7644ff2582114.jpg: ignoring corrupt image/label: labels mix segment and detection rows
albumentations: Affine(p=0.5, balanced_scale=False, border_mode=0, fill=0.0, fill_mask=0.0, fit_output=False, interpolation=1, keep_ratio=False, mask_interpolation=0, rotate=(0.0, 0.0), rotate_method='largest_box', scale={'x': (0.9, 1.1), 'y': (0.9, 1.1)}, shear={'x': (0.0, 0.0), 'y': (0.0, 

/home/himoza/miniconda3/lib/python3.13/site-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas
/home/himoza/miniconda3/lib/python3.13/site-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas
/home/himoza/miniconda3/lib/python3.13/site-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas



      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      1/200      4.14G      2.022      3.567    0.01171       1218        640: 6% ╸─────────── 4/58 2.4it/s 6.6s<22.6sWARNING ⚠️ CUDA out of memory in TaskAlignedAssigner with batch_size=12 and max_num_obj=1683; retrying assignment one image at a time on GPU. Model forward batch size is unchanged.
WARNING ⚠️ CUDA out of memory in TaskAlignedAssigner with batch_size=12 and max_num_obj=1683; retrying assignment one image at a time on GPU. Model forward batch size is unchanged.
      1/200      4.65G      1.863      3.343    0.00979        967        640: 18% ━━────────── 11/58 4.3it/s 8.3s<11.0s

...

      1/200      4.44G      1.216      1.542     0.0056        800        640: 100% ━━━━━━━━━━━━ 58/58 3.1it/s 19.0s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.4it/s 4.4s0.4ss
                   all        123       8279      0.882      0.898      0.933      0.678

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      2/200       4.5G      1.014     0.7495   0.004174       1039        640: 5% ╸─────────── 3/58 2.8it/s 0.9s<19.4s

/home/himoza/miniconda3/lib/python3.13/site-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas


      2/200      4.65G     0.9593     0.6025   0.004256       1040        640: 100% ━━━━━━━━━━━━ 58/58 4.3it/s 13.4s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 3.4it/s 1.8s0.4s
                   all        123       8279      0.898      0.902      0.937      0.682

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/200      4.43G     0.9963     0.5384   0.004463       1214        640: 100% ━━━━━━━━━━━━ 58/58 4.3it/s 13.5s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 3.5it/s 1.7s0.4s
                   all        123       8279      0.919      0.952      0.966      0.717

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      4/200      4.23G     0.9746     0.5102    0.00416       1288        640: 100% ━━━━━━━━━━━━ 58/58 4.2it/s 13.7s0.2s
                 Class     Images  Instance

/home/himoza/miniconda3/lib/python3.13/site-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas


      6/200      4.14G     0.9165     0.4428   0.004095        913        640: 100% ━━━━━━━━━━━━ 58/58 4.4it/s 13.3s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 3.8it/s 1.6s0.4s
                   all        123       8279      0.949       0.96      0.982      0.766

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      7/200      4.28G     0.8889     0.4201   0.003929        724        640: 100% ━━━━━━━━━━━━ 58/58 4.4it/s 13.2s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 3.8it/s 1.6s0.4s
                   all        123       8279      0.945      0.948      0.979      0.753

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      8/200      4.47G     0.8885     0.4231   0.003724       1009        640: 100% ━━━━━━━━━━━━ 58/58 4.3it/s 13.5s0.2s
                 Class     Images  Instance

### Валидация

In [13]:
metrics = validate_best_model(
      project_name,
      RESULTS_DIR,
      DATA_YAML,
      EPOCHS,
      BATCH_SIZE,
      IMGSZ,
    )

WARNING ⚠️ 'save_hybrid' is deprecated and will be removed in the future.


Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
YOLO26s summary (fused): 120 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1942.9±564.2 MB/s, size: 58.6 KB)
val: Scanning /home/himoza/block_detect/block-detection-1/valid/labels.cache... 123 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 123/123 64.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.0s/it 5.2s1.6ss
                   all        123       8279      0.964      0.967      0.989      0.834
Speed: 3.0ms preprocess, 25.9ms inference, 0.0ms loss, 2.8ms postprocess per image
Saving /home/himoza/block_detect/train_result/aug_yolo26s_training/val/predictions.json...

Evaluating faster-coco-eval mAP...
Evaluate annotation type *bbox*
COCOeval_opt.evaluate() finished...
DONE (t=0.12s).
Accumulating evaluation results...
COCOeval_opt.accumulate() fi

In [14]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    num_samples=30,
    conf_threshold=0.25,
    iou_threshold=0.45,
    img_size=640,
    seed=42
)

block-detection-1/valid/images
Обработка 1/30: 6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg


  Сохранено: comparison_001_6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  GT: 24 объектов, Predictions: 24 объектов

Обработка 2/30: artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  Сохранено: comparison_002_artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  GT: 36 объектов, Predictions: 36 объектов

Обработка 3/30: wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  Сохранено: comparison_003_wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  GT: 40 объектов, Predictions: 40 объектов

Обработка 4/30: 4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  Сохранено: comparison_004_4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  GT: 104 объектов, Predictions: 104 объектов

Обработка 5/30: _barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a75454ad8649138.jpg
  Сохранено: comparison_005__barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a75454ad8

'./train_result/aug_yolo26s_training/val/res'

## YOLOv26m

### Конфигурация

In [6]:
model = 'yolo26m.pt'
project_name = "aug_yolo26m_training"


BATCH_SIZE = 16

### Обучение

In [16]:
train_results = train_yolo(
    model,
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    EPOCHS,
    BATCH_SIZE,
    IMGSZ,
    PATIENCE,
    custom_augmentations
  )


НАЧАЛО ОБУЧЕНИЯ: aug_yolo26m_training
Время начала: 2026-09-30 23:07:54
Путь сохранения: /home/himoza/block_detect/train_result/aug_yolo26m_training
Эпохи: 200 | Patience: 15

New https://pypi.org/project/ultralytics/8.4.170 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[{'__version__': '2.0.8', 'transform': {'__class_fullname__': 'Affine', 'p': 0.5, 'balanced_scale': False, 'border_mode': 0, 'fill': 0.0, 'fill_mask': 0.0, 'fit_output': False, 'interpolation': 1, 'keep_ratio': False, 'mask_interpolation': 0, 'rotate': (0.0, 0.0), 'rotate_method': 'largest_box', 'scale': {'x': (0.9, 1.1), 'y': (0.9, 1.1)}, 'shear': {'x': (0.0, 0.0), 'y': (0.0, 0.0)}, 'translate_percent': {'x': (-0.03, 0.03), 'y': (-0.03, 0.03)}, 'translate_px': None}}, {'__version__': '2.0.8', 'transform': {'__class_fullname__': 'Ho

/home/himoza/miniconda3/lib/python3.13/site-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas
/home/himoza/miniconda3/lib/python3.13/site-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas
/home/himoza/miniconda3/lib/python3.13/site-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas
/home/himoza/miniconda3/lib/python3.13/site-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas


WARNING ⚠️ CUDA out of memory with batch=16. Reducing to batch=8 and retrying (1/3).
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1787.5±558.1 MB/s, size: 66.2 KB)
train: Scanning /home/himoza/block_detect/block-detection-1/train/labels.cache... 698 images, 0 backgrounds, 2 corrupt: 100% ━━━━━━━━━━━━ 698/698 154.1Mit/s 0.0s
train: /home/himoza/block_detect/block-detection-1/train/images/images124_jpg.rf.17c9b16c193a70c0399325fd995eda13.jpg: ignoring corrupt image/label: labels mix segment and detection rows
train: /home/himoza/block_detect/block-detection-1/train/images/images163_jpg.rf.1933bcd0ff9749f4c5f7644ff2582114.jpg: ignoring corrupt image/label: labels mix segment and detection rows
albumentations: Affine(p=0.5, balanced_scale=False, border_mode=0, fill=0.0, fill_mask=0.0, fit_output=False, interpolation=1, keep_ratio=False, mask_interpolation=0, rotate=(0.0, 0.0), rotate_method='largest_box', scale={'x': (0.9, 1.1), 'y': (0.9, 1.1)}, shear={'x': (0.0, 0.0), 'y': (0.0, 0

/home/himoza/miniconda3/lib/python3.13/site-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas



      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      1/200      4.83G      1.953       2.97    0.01379        521        640: 0% ──────────── 0/87  4.0sWARNING ⚠️ CUDA out of memory in TaskAlignedAssigner with batch_size=8 and max_num_obj=606; retrying assignment one image at a time on GPU. Model forward batch size is unchanged.
WARNING ⚠️ CUDA out of memory in TaskAlignedAssigner with batch_size=8 and max_num_obj=606; retrying assignment one image at a time on GPU. Model forward batch size is unchanged.
      1/200      4.86G      2.151      3.489    0.01228        645        640: 6% ╸─────────── 6/87 3.1it/s 6.0s<26.5s

...

/home/himoza/miniconda3/lib/python3.13/site-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas


      4/200      4.86G      0.978     0.5443    0.00452        934        640: 59% ━━━━━━━───── 52/87 3.5it/s 15.6s<9.9ss

/home/himoza/miniconda3/lib/python3.13/site-packages/albumentations/augmentations/dropout/functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas


      4/200      4.86G     0.9919     0.5246    0.00458        561        640: 100% ━━━━━━━━━━━━ 87/87 3.4it/s 25.8s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 3.3it/s 2.4s0.4s
                   all        123       8279       0.94      0.951      0.972      0.706

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      5/200      4.88G     0.9705      0.471   0.004398        680        640: 100% ━━━━━━━━━━━━ 87/87 3.3it/s 26.0s0.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 3.4it/s 2.4s0.3s
                   all        123       8279       0.95      0.948      0.981      0.744

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      6/200      4.87G     0.9153     0.4458   0.004154        375        640: 100% ━━━━━━━━━━━━ 87/87 3.4it/s 25.5s0.3ss
                 Class     Images  Instan

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 5.60 GiB of which 21.56 MiB is free. Including non-PyTorch memory, this process has 4.95 GiB memory in use. Of the allocated memory 4.48 GiB is allocated by PyTorch, and 293.88 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

### Валидация

In [7]:
metrics = validate_best_model(
      project_name,
      RESULTS_DIR,
      DATA_YAML,
      EPOCHS,
      BATCH_SIZE,
      IMGSZ,
    )

WARNING ⚠️ 'save_hybrid' is deprecated and will be removed in the future.
Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
YOLO26m summary (fused): 130 layers, 20,350,223 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2805.6±699.6 MB/s, size: 80.2 KB)
val: Scanning /home/himoza/block_detect/block-detection-1/valid/labels.cache... 123 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 123/123 36.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.1it/s 7.2s0.8ss
                   all        123       8279      0.956      0.966      0.985      0.818
Speed: 1.8ms preprocess, 36.1ms inference, 0.6ms loss, 6.4ms postprocess per image
Saving /home/himoza/block_detect/train_result/aug_yolo26m_training/val-2/predictions.json...

Evaluating faster-coco-eval mAP...
Evaluate annotation type *bbox*
COCOeval_opt.evaluate() finished...
DO